In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # makes results reproducible - same "random" data every run

n_players = 50
n_attempts_per_player = 40
rows = []

for player_id in range(n_players):
    # each player has a hidden "skill" level between 0 and 1
    skill = np.random.beta(2, 2)  # beta gives us realistic skill distribution (most players average, few extreme)
    
    difficulty = 1  # every player starts at difficulty 1
    recent_results = []  # track last 5 outcomes for rolling avg

    for attempt in range(n_attempts_per_player):
        # probability of success depends on skill vs difficulty
        success_prob = skill - (difficulty * 0.07) + 0.5
        success_prob = np.clip(success_prob, 0.05, 0.95)
        
        success = np.random.rand() < success_prob
        
        time_taken = np.random.normal(loc=60 - skill*20 + difficulty*3, scale=10)
        time_taken = max(5, time_taken)
        
        hints_used = np.random.poisson(lam=max(0, (1-skill)*2 + difficulty*0.2))
        prev_attempts = np.random.poisson(lam=1) if not success else 0
        
        avg_success_last_5 = np.mean(recent_results) if recent_results else 0.5
        
        rows.append({
            "player_id": player_id,
            "attempt_num": attempt,
            "difficulty_level": difficulty,
            "time_taken_sec": round(time_taken, 1),
            "hints_used": hints_used,
            "prev_attempts_this_puzzle": prev_attempts,
            "avg_success_rate_last_5": round(avg_success_last_5, 2),
            "success": int(success)
        })
        
        recent_results.append(int(success))
        if len(recent_results) > 5:
            recent_results.pop(0)
        
        # adaptive-ish difficulty change for NEXT attempt (simulating a simple existing system)
        if success:
            difficulty = min(10, difficulty + 1)
        else:
            difficulty = max(1, difficulty - 1)

df = pd.DataFrame(rows)
df.head(10)

,player_id,attempt_num,difficulty_level,time_taken_sec,hints_used,prev_attempts_this_puzzle,avg_success_rate_last_5,success
0,0,0,1,53.5,1,0,0.50,1
1,0,1,2,63.8,1,0,1.00,0
2,0,2,1,53.1,1,0,0.50,1
3,0,3,2,34.6,0,0,0.67,1
4,0,4,3,49.7,1,0,0.75,1
5,0,5,4,38.3,1,0,0.80,1
6,0,6,5,56.7,0,0,0.80,1
7,0,7,6,75.2,1,0,1.00,1
8,0,8,7,76.9,2,0,1.00,1
9,0,9,8,59.5,6,0,1.00,1


In [5]:
df.shape

(2000, 8)